# Pipeline de Visão Computacional de Produção
**Otimizado para Velocidade + Geração de Dados Vetoriais (JSON)**

Este notebook utiliza o modelo **YOLO-Seg** (Detecção + Segmentação Simultânea) eliminando o gargalo do SAM pesado.
Além disso, ele exporta um arquivo `.json` com o rastro (path) de todos os jogadores identificados.

In [ ]:
!pip install ultralytics supervision easyocr opencv-python scikit-learn
import cv2
import numpy as np
import supervision as sv
import easyocr
import json
from collections import defaultdict, Counter
from ultralytics import YOLO

print("Carregando modelos...")
yolo_seg_model = YOLO('yolov8x-seg.pt') # Modelo 2-em-1 (Detecta e Recorta na mesma passada)
reader = easyocr.Reader(['en'], gpu=True)
tracker = sv.ByteTrack()

In [ ]:
VIDEO_PATH = "../data/raw/videos/futebol_teste.mp4"
OUTPUT_PATH = "../data/raw/videos/futebol_rastreado_yoloseg.mp4"

ellipse_annotator = sv.EllipseAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_thickness=1, text_scale=0.5)

TEAM_NAMES = {0: "Brasil", 1: "Marrocos"}
tracker_jersey_memory = defaultdict(list)
tracker_team_memory = defaultdict(list)
trajectories_data = defaultdict(list) # NOSSO EXPORTADOR JSON

def extract_dominant_color(image, k=2):
    pixels = image.reshape((-1, 3))
    pixels = np.float32(pixels)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
    _, labels, centers = cv2.kmeans(pixels, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    counts = np.bincount(labels.flatten())
    return np.uint8(centers[np.argmax(counts)])

def get_team_id(color):
    b, g, r = color
    if r > 100 and g > 100: 
        return 0
    else:
        return 1

def process_frame(frame: np.ndarray, frame_index: int) -> np.ndarray:
    # O YOLO-Seg já devolve a detecção e a máscara pronta num piscar de olhos!
    results = yolo_seg_model(frame, classes=[0], conf=0.4, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = tracker.update_with_detections(detections)
    
    if len(detections) == 0:
        return frame
    
    final_labels = []
    for i in range(len(detections)):
        tracker_id = detections.tracker_id[i]
        x1, y1, x2, y2 = detections.xyxy[i].astype(int)
        
        bottom_center_x = int((x1 + x2) / 2)
        bottom_center_y = int(y2)
        player_crop = frame[y1:y2, x1:x2]
        
        if player_crop.size > 0:
            if len(tracker_team_memory[tracker_id]) < 10:
                # Usa a máscara perfeita do YOLO-Seg para ignorar a grama
                mask = detections.mask[i]
                mask_crop = mask[y1:y2, x1:x2]
                masked_player = cv2.bitwise_and(player_crop, player_crop, mask=mask_crop.astype(np.uint8))
                dominant_color = extract_dominant_color(masked_player)
                team_id = get_team_id(dominant_color)
                tracker_team_memory[tracker_id].append(team_id)
            
            if len(tracker_jersey_memory[tracker_id]) < 5:
                if player_crop.shape[0] > 40 and player_crop.shape[1] > 20: 
                    ocr_results = reader.readtext(player_crop, allowlist='0123456789')
                    for (bbox_ocr, text, prob) in ocr_results:
                        if prob > 0.3:
                            tracker_jersey_memory[tracker_id].append(text)
        
        if len(tracker_team_memory[tracker_id]) > 0:
            final_team_id = Counter(tracker_team_memory[tracker_id]).most_common(1)[0][0]
            team_name = TEAM_NAMES.get(final_team_id, "Desconhecido")
        else:
            team_name = "Desconhecido"

        if len(tracker_jersey_memory[tracker_id]) > 0:
            final_jersey = Counter(tracker_jersey_memory[tracker_id]).most_common(1)[0][0]
            label = f"Camisa {final_jersey} {team_name}"
        else:
            label = f"Jogador ({team_name})"
            
        final_labels.append(label)
        # Grava os passos para o Front-end
        trajectories_data[label].append({"frame": frame_index, "x": bottom_center_x, "y": bottom_center_y})

    annotated_frame = ellipse_annotator.annotate(scene=frame.copy(), detections=detections)
    annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=final_labels)
    
    return annotated_frame

sv.process_video(source_path=VIDEO_PATH, target_path=OUTPUT_PATH, callback=process_frame)

json_path = "../data/raw/videos/trajetorias.json"
with open(json_path, "w") as f:
    json.dump(trajectories_data, f)

print("Processamento concluído! Baixando arquivos...")
